# RecoverKv on LongGenBench-16K (branch `Clustered_int2_paused_fast`)

**Start to end, this notebook:**
1. runs **our method** (cache budget 20%, window 8, read-gate ratio 25%, 5 sink tokens; local 128 and int2 share 0.70
   in bytes mode, as in your efficiency notebook) on Llama-3.1-8B-Instruct;
2. **generates** LongGenBench-16K documents exactly as the runs behind the paper's Table 2 did;
3. **judges** them with the paper's judge under **vLLM**, batched, in one pass (minutes);
4. **scores** them with the paper's three metrics, **CR, STIC-1 and STIC-2**, plus Table 2's Len and wAvg, and shows
   the table.

**The benchmark.** LongGenBench (Wu et al., ICLR'25). A ~700-token prompt asks for one long structured document of up
to 16,000 tokens:
* 52 diary weeks, 52 weekly menus, a 100-floor skyscraper, or a 100-block city;
* with instructions that must surface in particular entries, once, over a range, or periodically.

The prompt is tiny, so almost everything the model attends to is its own output. At your settings the int2 tier and
the read gate start about 400 generated tokens in, and keys are first dropped about 8,900 tokens in.

**"As the paper did" means the code that produced Table 2**, read from the authors' repository history
(`mozhu621/LongGenBench`). The Llama-3.1-8B row first appears on 2024-08-18; this is the code in force then:

| step | reference code | here |
|---|---|---|
| prompt | `inference.py` @ `7b59ec6`: the dataset prompt as a raw string (no chat template). It ends with the first entry's header, `*** started ***\n#*# Week 1 (…):` | same; one BOS |
| decoding | `temperature=0.95, top_p=0.95, max_tokens=16000, seed=6211027` | same (top_k off, one seed for every document, as vLLM's per-request seed) |
| stop / output | `stop='*** finished'` (excluded); `output = prefix + text`, split on `#*#` | same |
| judge | `eval.py` @ `951769c`: Meta-Llama-3.1-8B-Instruct under vLLM, zero-shot `"Context: " + entry + "### Instruction: Does this context include the X? Please answer with 'yes' or 'no' only."`, `temperature=0.95, top_p=0.95, max_tokens=50, seed=42`, verdict `'yes' in response` | same, vLLM, character for character |
| metrics | CR (`eval.py`), STIC-1 = yes ÷ judged, STIC-2 = yes ÷ all instructions (`static.ipynb`), wAvg = CR × STIC-2 | same; the formulas re-derive 16 of 18 Table 2 rows from the authors' counts, digit for digit |

Where the paper's text disagrees with its code, the code wins:
* The paper says "greedy decoding", but every run sampled. `DECODING = "greedy"` exists if you want it.
* Today's `eval.py` has a few-shot prompt and the 70B judge. Both arrived in February 2025, after the paper.

The one unavoidable difference: our method lives in transformers' cache, so generation runs through HF `generate` at
batch size 1 in fp16, not vLLM in bf16.

**The judge and the scorer are a separate harness**, `longgenbench_eval/` in the repository. It imports nothing from
RecoverKv, so you can score any other baseline's generations with it the same way (its `README.md` has the format and
commands).

**Why a number here can be trusted:**
* `read_gate.verdict == "gated"` with `gated == fired > 0` in every part, and every cache knob is re-read from the
  sidecar and compared with step 8;
* the data's SHA-256 is checked, and its instruction count (6,205) equals the authors' table;
* the judge must pass a known-answer selftest, and every verdict file records the hash of the generations it judged.

**Before step 1:**
* *Runtime → Change runtime type → A100 GPU.*
* Accept the licence at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, then add your HF token as a Colab
  secret named `HF_TOKEN` (key icon in the left bar) and allow this notebook to read it.

### How long it takes (one A100)
These are arithmetic estimates, not measurements. They assume ~13,000 new tokens per document (Llama-3.1-8B averages
~8,800 words at 16K in the paper) and ~55 ms/token for ours at batch size 1, or ~30 ms/token for the optional full-KV
arm. Step 11 re-projects the rest from the measured speed after every part.

| `N_PER_TYPE` | documents | ours | + full KV (`RUN_FULLKV = True`) | judge + score (vLLM) |
|---|---|---|---|---|
| **5** (default) | 20 | **~4 h** | +2.2 h | ~3 min |
| 10 | 40 | ~8 h | +4.3 h | ~4 min |
| 100 (the whole benchmark) | 400 | ~80 h | +43 h | ~15 min |

The paper ran all 400 documents with vLLM on eight GPUs. At batch size 1 through our cache that is days, so a Colab run
takes the first `N_PER_TYPE` documents of each type, the same documents in every arm. Raising `N_PER_TYPE` in
multiples of `CHUNK` reuses every part already done, and a Colab disconnect costs at most the part that was running.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

## 2. Clone the branch, pinned
This pins `461e35c`, the commit with this runner and the `longgenbench_eval` harness. Your efficiency notebook pins
`7beb3da`. Between the two, two commits touched cache code:
* `db9a7ed` adds the completion-ablation knob. It is off by default, and every part is checked for that.
* `5924f03` moves the int2 quantiser's trace gate from torch 2.7 to 2.8. That is a no-op on the torch 2.6 installed here.

If you quote speed and accuracy side by side, say they come from adjacent commits.

In [ ]:
%cd /content
!rm -rf RecoverKv
!git clone -q --branch Clustered_int2_paused_fast --single-branch https://github.com/shaswata965-coder/RecoverKv.git
%cd /content/RecoverKv
!git checkout -q 461e35cf7af14ad877b46dbcad5844b9a01aaf2e && git log -1 --format='%h %s'
!test -f longgenbench_eval/judge.py && test -f modules/evaluation/longgenbench_runner.py && echo "runner and harness present"

## 3. Install both stacks (~8 min)
* **Generation** (this notebook's Python): torch 2.6, transformers exactly 4.47.1 (`utils/cache_factory.py` refuses
  anything newer), and flash-attn from its prebuilt wheel. If pip starts *compiling* flash-attn, stop the cell: no
  wheel matched this Python.
* **Judge** (its own virtualenv, `/content/judge_env`): `vllm==0.8.5.post1` from a fully pinned lock
  (`longgenbench_eval/requirements-judge.lock.txt`, resolved as of vLLM's release in May 2025).
  * Why its own environment: vLLM needs transformers ≥ 4.51, which the cache refuses, so the two stacks live apart
    and never import each other.
  * Why a lock: vLLM 0.8.5's dependencies are open-ended, and resolved today they pull in transformers 5.x, which
    vLLM 0.8.5 cannot load a tokenizer with.

**When this cell finishes: Runtime → Restart session**, then continue at step 4.

In [ ]:
%%bash
set -euo pipefail
pip -q install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

# flash-attn: prebuilt wheel matching this python / torch 2.6 / C++ ABI
read PYV ABI < <(python -c "import sys,torch; print(f'cp{sys.version_info[0]}{sys.version_info[1]}', 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')")
ok=0
for V in 2.7.4.post1 2.8.3; do
  WHL="https://github.com/Dao-AILab/flash-attention/releases/download/v${V}/flash_attn-${V}+cu12torch2.6cxx11abi${ABI}-${PYV}-${PYV}-linux_x86_64.whl"
  echo "trying $WHL"
  if pip -q install --no-deps "$WHL"; then ok=1; break; fi
done
[ "$ok" = 1 ] || { echo "no prebuilt flash-attn wheel for ${PYV}/torch2.6/abi${ABI}"; exit 1; }
pip -q install "transformers==4.47.1" "tokenizers>=0.21,<0.22" "huggingface_hub>=0.24,<1.0" \
               "datasets>=2.16" "accelerate>=0.26" sentencepiece protobuf pyyaml einops pytest uv
python -c "import torch, transformers, flash_attn; print('generation stack: torch', torch.__version__, '| transformers', transformers.__version__, '| flash_attn', flash_attn.__version__)"

# The judge: vLLM in its own virtualenv (Python 3.12: vLLM 0.8.5 supports < 3.13; uv fetches 3.12 if needed).
cd /content/RecoverKv
rm -rf /content/judge_env          # a fresh env every time: the lock is the whole truth
uv venv -q --python 3.12 /content/judge_env
uv pip install -q --python /content/judge_env/bin/python -r longgenbench_eval/requirements-judge.lock.txt
env -u PYTHONPATH /content/judge_env/bin/python -c "import vllm, torch, transformers; assert transformers.__version__ == '4.51.3', transformers.__version__; print('judge stack: vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__)"
echo "NOW: Runtime -> Restart session, then run step 4."

## 4. Hugging Face login + model download (~16 GB; the judge reuses the same files)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, snapshot_download
login(token=userdata.get("HF_TOKEN"))      # also writes the token file the judge's vLLM reads
snapshot_download("meta-llama/Llama-3.1-8B-Instruct",
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])  # skips the 16 GB original/*.pth

## 5. LongGenBench data, pinned
`Dataset_short.json` (16K) and `Dataset_long.json` (32K) from `mozhu/LongGenBench` at revision `3c96cac`. Each file's
SHA-256 is checked, and its instruction count must equal the authors' published total.

In [ ]:
%%bash
set -euo pipefail
cd /content/RecoverKv
python -m longgenbench_eval.data --out data/longgenbench

## 6. Preflight
It must print `PREFLIGHT OK`. It checks:
* **Generation side:** the transformers pin, the flash-attn LSE that L-reuse needs, Inductor compiling, the read gate
  having a production call site, and the shipped configs being at the operating point. It also checks one BOS on a
  raw prompt, `stop_strings` support, and the model's EOS ids (what vLLM stops on).
* **Harness tests:** the CPU tests that pin the runner and the harness to the reference code, including the
  re-derivation of the paper's Table 2.
* **Judge side:** vLLM imports in its own environment, with the pinned transformers, and its tokenizer loads through
  the same code path `LLM(...)` takes (where an unpinned environment breaks).

In [ ]:
%%bash
set -euo pipefail
cd /content/RecoverKv
python - <<'EOF'
import re, subprocess, sys, torch, transformers, triton, flash_attn
print("python", sys.version.split()[0], "| torch", torch.__version__, "cuda", torch.version.cuda,
      "| triton", triton.__version__, "| transformers", transformers.__version__, "| flash_attn", flash_attn.__version__)
assert torch.__version__.startswith("2.6"), "torch must be 2.6.x"
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()                     # == 4.47.1
p = torch.cuda.get_device_properties(0)
print(f"gpu {p.name}  {p.total_memory / 2**30:.1f} GiB")
from flash_attn import flash_attn_func                       # L-reuse needs the LSE back
q = torch.randn(1, 128, 32, 128, device="cuda", dtype=torch.float16)
k = torch.randn(1, 128, 8, 128, device="cuda", dtype=torch.float16)
out = flash_attn_func(q, k, k, causal=True, return_attn_probs=True)
assert isinstance(out, tuple) and tuple(out[1].shape) == (1, 32, 128), "flash-attn returned no LSE"
f = torch.compile(lambda x: (x * 2).sin().sum(-1))             # the eviction is compiled-or-raise
f(torch.randn(64, 64, device="cuda")); torch.cuda.synchronize()
grep = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'", shell=True,
                      capture_output=True, text=True).stdout.splitlines()
calls = [l for l in grep if re.search(r"=\s*fused_gate\(", l)]
assert calls, "fused_gate has no non-test call site: the decode step would not be gated"
print("read gate call site:", ":".join(calls[0].split(":")[:2]))
r = subprocess.run([sys.executable, "scripts/check_operating_point.py"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout[-3000:]
from transformers import AutoTokenizer, GenerationConfig
from utils.prompting import encode_prompt
from longgenbench_eval import data as D
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
doc = D.load_type("data/longgenbench", "16K", "Week", 0, 1)[0]
ids = encode_prompt(tok, doc["prompt"], return_tensors="pt").input_ids[0].tolist()
assert ids[0] == tok.bos_token_id and ids[1] != tok.bos_token_id, ids[:4]
assert hasattr(GenerationConfig(), "stop_strings"), "this transformers has no stop_strings"
eos = GenerationConfig.from_pretrained("meta-llama/Llama-3.1-8B-Instruct").eos_token_id
assert sorted(eos) == [128001, 128008, 128009], eos
print("generation side OK")
EOF
LONGGENBENCH_DATA=data/longgenbench python -m pytest -q -p no:cacheprovider tests/test_longgenbench.py tests/test_longgenbench_eval.py
env -u PYTHONPATH VLLM_USE_V1=0 /content/judge_env/bin/python - <<'EOF'
import vllm, transformers
from vllm.transformers_utils.tokenizer import get_tokenizer
assert transformers.__version__ == "4.51.3", transformers.__version__
tok = get_tokenizer("meta-llama/Llama-3.1-8B-Instruct")      # the call LLM(...) makes; crashed when unpinned
print("judge side: vllm", vllm.__version__, "| transformers", transformers.__version__, "|", type(tok).__name__,
      "| eos", tok.eos_token)
EOF
echo "PREFLIGHT OK"

## 7. Mount Google Drive
Every generation part, the verdicts and the final table are copied here as soon as they exist, and earlier verified
parts are reused from here, so nothing is lost if the runtime is recycled.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 8. Settings (the only cell you edit)
The method settings are yours: budget 20%, window 8, read-gate ratio 25%, 5 sink tokens, plus local 128 and int2 share
0.70 in bytes mode from your efficiency notebook. These are the shipped operating point (`utils.config.OPERATING_POINT`).

The budget is `CACHE_BUDGET × (prompt + max_new_tokens)`, the method's own definition (your
`BUDGET_BASIS = "prefill_plus_gen"`). At 16K that is about 3,300 tokens' worth of bytes.

* `N_PER_TYPE`: the first N documents of each type, the same in every arm. Keep it a multiple of `CHUNK`.
* `RUN_FULLKV`: also generate the full-KV baseline on the same documents (about +55% time); the table then shows
  ours − full KV with a paired interval.
* `DECODING`: `"reference"` (the sampling of the runs behind Table 2) or `"greedy"` (what the paper's text says).
* `JUDGE_*`: the paper's judge by default. Change them only for a sensitivity check.

In [ ]:
# ---- our method (your configuration) ----
CACHE_BUDGET = 0.20            # 20% of (prompt + max_new_tokens), in bytes
WINDOW_SIZE  = 8
GATE_RATIO   = 0.25            # fraction of int2 windows the read gate opens per step
NUM_SINK     = 5
LOCAL_WINDOW = 128             # int multiple of WINDOW_SIZE
QUANT_RATIO  = 0.70            # int2 share of the byte budget
QUANT_MODE   = "bytes"
DTYPE        = "float16"
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
# ---- benchmark ----
SETTING      = "16K"           # "16K" (up to 16,000 new tokens) | "32K" (32,000)
TYPES        = None            # None = all four: Week, Floor, Menu Week, Block
N_PER_TYPE   = 5               # first N documents of each type; 100 = the whole benchmark
CHUNK        = 5               # documents per generation part
DECODING     = "reference"     # "reference" (Table 2's runs: T=0.95, top_p=0.95) | "greedy"
RUN_FULLKV   = False           # True: also the full-KV baseline on the same documents
CONTROL_ROW  = False           # True: also ours at quant_gate_ratio 1.0 (the gate's control arm)
# ---- judge (the paper's) ----
JUDGE_MODEL  = "meta-llama/Llama-3.1-8B-Instruct"
JUDGE_PROMPT = "paper"         # "paper" (eval.py @ 951769c) | "2025" (later few-shot prompt)
JUDGE_GREEDY = False           # False = the paper's T=0.95, top_p=0.95, seed 42
# ---- bookkeeping ----
REUSE_VERIFIED = True
READ_FRACTION_TOL = 0.05       # warn if the realised read fraction is further than this from GATE_RATIO

## 9. Helpers (run as-is)

In [ ]:
import json, os, shutil, subprocess, sys, time, importlib, hashlib
from pathlib import Path
import numpy as np

REPO = "/content/RecoverKv"
PIN = "461e35cf7af14ad877b46dbcad5844b9a01aaf2e"
JUDGE_PY = "/content/judge_env/bin/python"
sys.path.insert(0, REPO)
import longgenbench_eval.reference as R, longgenbench_eval.data as D, longgenbench_eval.metrics as M
import longgenbench_eval.judge as J, longgenbench_eval.score as SC
for _m in (R, D, M, J, SC):
    importlib.reload(_m)

PY = shutil.which("python") or sys.executable
DATA_DIR = f"{REPO}/data/longgenbench"
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
OUT_ROOT = Path(f"{REPO}/outputs/lgb16")
DRIVE_MOUNT = Path("/content/drive/MyDrive")
DRIVE_ROOT = DRIVE_MOUNT / "recoverkv_longgenbench16k"
CARD_DEFAULT = {"mu": 4, "v": 8, "t": 8, "vm": 4}       # the shipped gate card
LLAMA31_EOS = [128001, 128008, 128009]                  # generation_config eos: what vLLM stops on
SMOKE_TYPE, SMOKE_NEW = "Menu Week", 2048               # its first document has instructions on weeks 3 and 4
TOK_PER_DOC = 13000
SEC_PER_TOKEN = {"fullkv": 0.030, "ours": 0.055, "ctrl_gate1.0": 0.060}   # priors; replaced by measurements


class PartFailed(Exception):
    pass


# ----------------------------------------------------------------- settings and plan
def types():
    return list(TYPES) if TYPES else list(R.TYPES[SETTING])


def _git_head():
    r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
    return r.stdout.strip() or "no_git"


def _gpu_name():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    except OSError:
        return None
    return r.stdout.strip().splitlines()[0] if r.stdout.strip() else None


def check_knobs():
    assert SETTING in R.SETTINGS, f"SETTING must be one of {R.SETTINGS}"
    for t in types():
        D.check_type(SETTING, t)
    assert int(N_PER_TYPE) >= 1 and int(CHUNK) >= 1
    assert DECODING in ("reference", "greedy") and JUDGE_PROMPT in R.JUDGE_PROMPTS
    assert 0.0 < QUANT_RATIO <= 1.0, "QUANT_RATIO in (0, 1]: the method has an int2 tier and a read gate"
    assert 0.0 < GATE_RATIO <= 1.0 and 0.0 < CACHE_BUDGET <= 1.0 and QUANT_MODE in ("bytes", "tokens")
    assert DTYPE in ("float16", "bfloat16") and WINDOW_SIZE % 4 == 0
    assert isinstance(LOCAL_WINDOW, int) and LOCAL_WINDOW % WINDOW_SIZE == 0, "LOCAL_WINDOW: int multiple of WINDOW_SIZE"
    assert _git_head() == PIN, f"{REPO} is at {_git_head()[:10]}, not the pinned {PIN[:10]}: run step 2 again"
    D.dataset_sha(DATA_DIR, SETTING)                    # present and byte-identical to the pin


def arms():
    a = [("ours", float(GATE_RATIO))]
    if CONTROL_ROW and GATE_RATIO < 1.0:
        a.append(("ctrl_gate1.0", 1.0))
    if RUN_FULLKV:
        a.insert(0, ("fullkv", None))
    return a


def type_ids(t):
    return [d["id"] for d in D.load_type(DATA_DIR, SETTING, t, 0, int(N_PER_TYPE))]


def plan():
    parts = []
    for t in types():
        ids = type_ids(t)
        for c0 in range(0, len(ids), int(CHUNK)):
            c1 = min(c0 + int(CHUNK), len(ids))
            for arm, gate in arms():
                parts.append(dict(type=t, slug=D.type_slug(t), start=c0, end=c1, ids=ids[c0:c1], arm=arm, gate=gate))
    return parts


def op_tag():
    return f"b{CACHE_BUDGET}_q{QUANT_RATIO}_w{WINDOW_SIZE}_s{NUM_SINK}_l{LOCAL_WINDOW}_{QUANT_MODE}_{DTYPE}"


def arm_dir(arm, gate, smoke=False):
    sub = f"fullkv_{DTYPE}" if arm == "fullkv" else f"{arm}_g{gate}_{op_tag()}"
    return OUT_ROOT / ("smoke_" + SETTING if smoke else SETTING) / sub / DECODING


def part_dir(p, smoke=False):
    return arm_dir(p["arm"], p["gate"], smoke) / f"{p['slug']}_{p['start']:03d}-{p['end']:03d}"


def part_settings(p, max_new=None):
    s = dict(setting=SETTING, type=p["type"], start=p["start"], end=p["end"], ids=p["ids"], arm=p["arm"],
             model=MODEL_ID, dtype=DTYPE, decoding=DECODING, max_new_tokens_override=max_new,
             dataset_revision=R.HF_REVISION, commit=_git_head())
    if p["arm"] != "fullkv":
        s.update(cache_budget=CACHE_BUDGET, window_size=WINDOW_SIZE, num_sink_tokens=NUM_SINK,
                 local_window_size=LOCAL_WINDOW, quant_ratio=QUANT_RATIO, quant_budget_mode=QUANT_MODE,
                 quant_gate_ratio=p["gate"], first_eviction_step=0)
    return s


def config_text(p, out, max_new=None):
    j = json.dumps
    lines = ["# generated by notebooks/longgenbench_colab.ipynb", "run: {mode: longgenbench, seed: 42}",
             f"model: {{name: {j(MODEL_ID)}, revision: null, dtype: {DTYPE}, attn_implementation: flash_attention_2}}"]
    if p["arm"] == "fullkv":
        lines += ["cache: {backend: dynamic, backend_package: null, cache_budget: null}"]
    else:
        lines += ["cache:", "  backend: windowed", "  backend_package: flash_attn",
                  f"  cache_budget: {float(CACHE_BUDGET)}", f"  window_size: {int(WINDOW_SIZE)}",
                  f"  num_sink_tokens: {int(NUM_SINK)}", f"  local_window_size: {int(LOCAL_WINDOW)}",
                  f"  quant_ratio: {float(QUANT_RATIO)}", f"  quant_budget_mode: {QUANT_MODE}",
                  f"  quant_gate_ratio: {float(p['gate'])}", "  first_eviction_step: 0",
                  f"window: {{window_size: {int(WINDOW_SIZE)}, num_sink_tokens: {int(NUM_SINK)}, "
                  f"local_window_size: {int(LOCAL_WINDOW)}}}"]
    lines += ["telemetry: {track_scores: false}", "longgenbench:", f"  data_dir: {j(DATA_DIR)}",
              f"  setting: {j(SETTING)}", f"  types: [{j(p['type'])}]", f"  output_dir: {j(str(out))}",
              f"  start: {p['start']}", f"  end: {p['end']}", f"  decoding: {DECODING}",
              f"  sample_seed: {R.GENERATION_SAMPLING['seed']}", "  skip_oom: true", "  aggressive_cache_clear: true",
              f"  max_new_tokens: {'null' if max_new is None else int(max_new)}", ""]
    return "\n".join(lines)


# ----------------------------------------------------------------- files
def _fresh(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
    return path


def _drive_path(local):
    return DRIVE_ROOT / Path(local).relative_to(OUT_ROOT)


def _to_drive(src):
    if not DRIVE_MOUNT.is_dir():
        return
    src = Path(src)
    dst = _fresh(_drive_path(src))
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
    print("copied to", dst)


def _restore(p, out):
    out = Path(out)
    if not (out / f"{p['slug']}.meta.json").is_file() and DRIVE_MOUNT.is_dir() and _drive_path(out).is_dir():
        shutil.copytree(_drive_path(out), out, dirs_exist_ok=True)
        print("  found in Drive:", out.relative_to(OUT_ROOT))
    return out


def _stream(cmd, log_path, keys, cwd=REPO, env=None):
    tail = []
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=cwd, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            tail = (tail + [line])[-40:]
            if any(k in line for k in keys):
                print("   ", line.rstrip()[:220])
        return proc.wait(), "".join(tail)


# ----------------------------------------------------------------- the cache geometry, before any GPU hour
GEO_PY = r"""
import json, sys, torch
from transformers import AutoConfig, AutoTokenizer
from utils.cache_factory import get_cache_classes
from utils.prompting import encode_prompt
from longgenbench_eval import data as D, reference as R
a = json.loads(sys.argv[1])
tok = AutoTokenizer.from_pretrained(a["model"]); mc = AutoConfig.from_pretrained(a["model"])
_, WCC, _ = get_cache_classes("flash_attn")
cc = WCC(window_size=a["w"], num_sink_tokens=a["s"], local_window_size=a["l"], cache_budget=a["b"],
         quant_ratio=a["q"], quant_budget_mode=a["mode"], quant_gate_ratio=a["g"], first_eviction_step=0)
out = {}
for t in a["types"]:
    n = [len(encode_prompt(tok, d["prompt"], return_tensors="pt").input_ids[0])
         for d in D.load_type(a["data"], a["setting"], t, 0, a["n"])]
    rows = {}
    for name, P in (("min", min(n)), ("max", max(n))):
        r = cc.resolve(P, mc, torch.float16, R.MAX_NEW_TOKENS[a["setting"]])
        rows[name] = dict(prompt=P, budget_tokens=r.total_budget_tokens, N_q=r.N_q,
                          fp_cap=r.num_sink_tokens + r.local_tokens + r.top_k_fp * r.window_size, cap=r.retained_tokens)
    out[t] = rows
print("GEO" + json.dumps(out))
"""


def geometry():
    arg = json.dumps(dict(model=MODEL_ID, w=WINDOW_SIZE, s=NUM_SINK, l=LOCAL_WINDOW, b=CACHE_BUDGET, q=QUANT_RATIO,
                          mode=QUANT_MODE, g=GATE_RATIO, types=types(), data=DATA_DIR, setting=SETTING, n=int(N_PER_TYPE)))
    r = subprocess.run([PY, "-c", GEO_PY, arg], cwd=REPO, env=ENV, capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("GEO")]
    if r.returncode or not line:
        raise SystemExit("cache geometry preview failed:\n" + r.stderr[-3000:])
    geo = json.loads(line[0][3:])
    from utils.config import OPERATING_POINT as OP
    mine = dict(cache_budget=CACHE_BUDGET, window_size=WINDOW_SIZE, num_sink_tokens=NUM_SINK,
                local_window_size=LOCAL_WINDOW, quant_ratio=QUANT_RATIO, quant_budget_mode=QUANT_MODE,
                quant_gate_ratio=GATE_RATIO)
    print(f"settings: budget {CACHE_BUDGET}, window {WINDOW_SIZE}, gate {GATE_RATIO}, sink {NUM_SINK}, "
          f"local {LOCAL_WINDOW}, int2 share {QUANT_RATIO} ({QUANT_MODE}), {DTYPE}")
    off = {k: (v, OP[k]) for k, v in mine.items() if v != OP[k]}
    if off:
        print(f"  !! NOT the shipped operating point (yours, shipped): {off}")
    print(f"our cache at these settings ({SETTING}, max_new_tokens {R.MAX_NEW_TOKENS[SETTING]}):")
    print(f"  {'type':10s} {'prompt':>11s} {'budget tok':>11s} {'fp16 keys':>10s} {'int2 win':>9s} {'keys kept':>10s} "
          f"{'int2 from':>10s} {'drops from':>11s}")
    for t, rows in geo.items():
        for name in ("min", "max"):
            g = rows[name]
            print(f"  {t:10s} {name + ' ' + str(g['prompt']):>11s} {g['budget_tokens']:>11d} {g['fp_cap']:>10d} "
                  f"{g['N_q']:>9d} {g['cap']:>10d} {max(0, g['fp_cap'] - g['prompt']):>10d} "
                  f"{max(0, g['cap'] - g['prompt']):>11d}")
        if rows["min"]["N_q"] == 0:
            raise SystemExit(f"{t}: no int2 window fits the budget (N_q = 0): no int2 tier, nothing to gate")
    print("  ('int2 from' / 'drops from' = generated tokens before the int2 tier starts filling / keys are dropped)")
    return geo


# ----------------------------------------------------------------- generation: one part = one run of main.py
def run_part(p, out, max_new=None):
    out = Path(out)
    _fresh(out)
    if DRIVE_MOUNT.is_dir():
        _fresh(_drive_path(out))
    out.mkdir(parents=True)
    cfg = out / "_longgenbench.generated.yaml"
    cfg.write_text(config_text(p, out, max_new))
    (out / "run_settings.json").write_text(json.dumps(part_settings(p, max_new), indent=2))
    t0 = time.time()
    rc, tail = _stream([PY, "main.py", "--config", str(cfg)], out / "run.log",
                       ("new tokens in", "read gate", "compression:", "never filled", "THE READ GATE",
                        "Error", "error:", "Traceback", "WARNING"))
    (out / "wall_s.txt").write_text(f"{time.time() - t0:.1f}\n")
    if rc:
        print(tail)
        kind = "ran out of GPU memory" if "out of memory" in tail.lower() else f"main.py exited with {rc}"
        raise PartFailed(f"generation {kind}; full log: {out}/run.log")
    return out


def _load(p, out):
    out = Path(out)
    meta = json.loads((out / f"{p['slug']}.meta.json").read_text())
    recs = [json.loads(l) for l in (out / f"{p['slug']}.jsonl").read_text().splitlines() if l.strip()]
    return meta, recs


def verify(p, out, max_new=None, quiet=False):
    # Every field a generation sidecar must carry for its documents to belong to this part, arm and protocol.
    fails, warns = [], []
    try:
        meta, recs = _load(p, out)
    except (OSError, ValueError) as e:
        return False, [f"generation incomplete: {e}"], []

    def eq(k, want, src=None, what=""):
        got = (meta if src is None else src).get(k)
        if got != want:
            fails.append(f"{what}{k} = {got!r}, expected {want!r}")

    eq("benchmark", "LongGenBench"); eq("setting", SETTING); eq("type", p["type"])
    eq("dataset_revision", R.HF_REVISION); eq("dataset_sha", R.FILES[SETTING][1])
    eq("prompt_format", R.PROMPT_FORMAT); eq("ids", p["ids"])
    eq("max_new_tokens", int(max_new or R.MAX_NEW_TOKENS[SETTING])); eq("max_new_tokens_override", max_new)
    eq("model_name", MODEL_ID); eq("dtype", DTYPE); eq("attn_implementation", "flash_attention_2")
    eq("transformers_version", "4.47.1")
    if meta.get("commit_sha") != PIN:
        fails.append(f"commit {str(meta.get('commit_sha'))[:10]} is not the pinned {PIN[:10]}")
    gen = meta.get("generation") or {}
    for k, v in (("decoding", DECODING), ("stop_string", R.STOP_STRING), ("stop_string_in_output", False),
                 ("eos_token_id", LLAMA31_EOS), ("output", "prefix + generation"), ("batch_size", 1)):
        eq(k, v, gen, "generation.")
    if DECODING == "reference":
        want_s = {**R.GENERATION_SAMPLING, "seed_rule": "the same seed for every document"}
        if gen.get("sampling") != want_s:
            fails.append(f"generation.sampling = {gen.get('sampling')!r}, expected {want_s!r}")
    if [r.get("id") for r in recs] != p["ids"]:
        fails.append("records do not cover exactly this part's documents")
    if sum(r.get("output") is None for r in recs):
        warns.append(f"{sum(r.get('output') is None for r in recs)} document(s) ran out of GPU memory (score 0)")
    rg = meta.get("read_gate") or {}
    if p["arm"] == "fullkv":
        eq("cache_type", "full_cache")
        if rg.get("verdict") != "not-expected":
            fails.append(f"full-cache read gate verdict {rg.get('verdict')!r}, expected 'not-expected'")
    else:
        eq("cache_type", "windowed"); eq("cache_backend_package", "flash_attn")
        for k, v in (("cache_budget", float(CACHE_BUDGET)), ("window_size", int(WINDOW_SIZE)),
                     ("num_sink_tokens", int(NUM_SINK)), ("local_window_size", int(LOCAL_WINDOW)),
                     ("quant_ratio", float(QUANT_RATIO)), ("quant_budget_mode", QUANT_MODE),
                     ("quant_gate_ratio", float(p["gate"])), ("first_eviction_step", 0),
                     ("quant_promotion", "bidir"), ("quant_promote_source", "dequant"),
                     ("quant_card_bits", CARD_DEFAULT)):
            eq(k, v)
        if (meta.get("quant_completion") or {}).get("mode") != "native":
            fails.append(f"quant_completion {meta.get('quant_completion')!r} is not the shipped kernel path")
        if rg.get("verdict") != "gated":
            fails.append(f"read gate verdict {rg.get('verdict')!r}, expected 'gated' "
                         f"(gated={rg.get('gated')} fired={rg.get('fired')})")
        if not rg.get("fired") or rg.get("gated") != rg.get("fired"):
            fails.append(f"gate ran on {rg.get('gated')} of {rg.get('fired')} fused layers")
        if rg.get("completion_applied"):
            fails.append("the completion-ablation epilogue ran: not the shipped method")
        rf = rg.get("read_fraction")
        if rf is None or abs(rf - float(p["gate"])) > READ_FRACTION_TOL:
            warns.append(f"realised read fraction {rf} vs configured {p['gate']}")
        geo = meta.get("resolved_geometry_first_example") or {}
        if not geo.get("N_q"):
            fails.append(f"NO int2 tier at this shape (N_q = {geo.get('N_q')}): nothing to gate")
        for k, v in (("window_size", int(WINDOW_SIZE)), ("num_sink_tokens", int(NUM_SINK)),
                     ("local_tokens", int(LOCAL_WINDOW)), ("quant_budget_mode", QUANT_MODE)):
            eq(k, v, geo, "geometry.")
        if meta.get("n_examples_no_int2"):
            warns.append(f"{meta['n_examples_no_int2']} document(s) never filled past the fp16 tier (no int2, no gate)")
    if not quiet:
        for w in warns:
            print("  warn:", w)
        for f in fails:
            print("  FAIL:", f)
    return not fails, fails, warns


def report_part(p, out, max_new=None):
    meta, recs = _load(p, out)
    rg = meta.get("read_gate") or {}
    stops = {k: sum(r.get("stopped_on") == k for r in recs) for k in ("stop_string", "eos", "max_new_tokens")}
    cr = np.mean([r.get("completion_rate", 0.0) for r in recs])
    ft = [r.get("first_token_s") for r in recs if r.get("first_token_s") is not None]
    ms = [r.get("decode_ms_per_token") for r in recs if r.get("decode_ms_per_token") is not None]
    line = (f"  {p['type']} [{p['start']}, {p['end']}) {p['arm']}: CR {cr:.1f}%  new tokens {meta.get('total_gen_tokens')}"
            f"  stopped on {stops}  peak {meta.get('peak_mem_gib_max')} GiB"
            f"\n    first token {', '.join(f'{x:.0f} s' for x in ft) or 'n/a'} (prefill + this process's compiles);"
            f" then {', '.join(f'{x:.0f}' for x in ms) or 'n/a'} ms/token")
    if p["arm"] != "fullkv":
        rf = rg.get("read_fraction")
        line += (f"\n    gate {rg.get('verdict')} ({rg.get('gated')}/{rg.get('fired')} layers, read "
                 f"{'n/a' if rf is None else f'{rf:.3f}'})  int2 unused on {meta.get('n_examples_no_int2')}/{len(recs)}"
                 f"  no key dropped on {meta.get('n_examples_no_eviction')}/{len(recs)}  KV bytes <= "
                 f"{meta.get('mean_kv_bytes_fraction_upper')} of full")
    wall = Path(out) / "wall_s.txt"
    if wall.is_file():
        line += f"\n    {float(wall.read_text()) / 60:.1f} min"
    print(line)
    ok, _, _ = verify(p, out, max_new)
    print("  VERIFIED" if ok else "  NOT VERIFIED: this part will not be scored")
    return ok


def _reusable(p, out):
    if not REUSE_VERIFIED:
        return False
    out = _restore(p, out)
    try:
        st = json.loads((out / "run_settings.json").read_text())
    except (OSError, ValueError):
        return False
    diff = {k: (st.get(k), v) for k, v in part_settings(p).items() if st.get(k) != v}
    if diff:
        print("  earlier result not reusable, settings differ (was, now):", diff)
        return False
    try:
        meta, _ = _load(p, out)
    except (OSError, ValueError):
        return False
    if meta.get("gpu_name") != _gpu_name():
        print(f"  earlier result not reusable: recorded on {meta.get('gpu_name')}, this GPU is {_gpu_name()}")
        return False
    return verify(p, out, quiet=True)[0]


def _projection(remaining, measured, tok_per_doc):
    return sum(len(p["ids"]) * tok_per_doc * (measured.get(p["arm"]) or SEC_PER_TOKEN[p["arm"]]) + 90.0
               for p in remaining) / 3600


def generate_all():
    check_knobs()
    geometry()
    parts = plan()
    print(f"\nGPU: {_gpu_name()} | commit: {_git_head()[:8]} | Drive: "
          f"{'mounted' if DRIVE_MOUNT.is_dir() else 'NOT mounted (results stay local)'}")
    print(f"{len(parts)} generation parts: {len(types())} types x {int(N_PER_TYPE)} documents x arms "
          f"{[a for a, _ in arms()]}; projected {_projection(parts, {}, TOK_PER_DOC):.1f} h if nothing is reused")
    failed, tok, sec, per_doc = [], {}, {}, []
    measured = {}
    for k, p in enumerate(parts):
        out = part_dir(p)
        print(f"\n#################### part {k + 1}/{len(parts)}: {SETTING} {p['type']} "
              f"[{p['start']}, {p['end']}) {p['arm']} ####################")
        if _reusable(p, out):
            print("  reusing the verified earlier result, not re-run")
            continue
        try:
            run_part(p, out)
        except PartFailed as e:
            failed.append(f"{p['type']} [{p['start']}, {p['end']}) {p['arm']}: {e}")
            print(f"\n!! {failed[-1]}\n   flagged; continuing with the next part")
            _to_drive(out)
            continue
        if not report_part(p, out):
            failed.append(f"{p['type']} [{p['start']}, {p['end']}) {p['arm']}: did not verify")
        _to_drive(out)
        meta, recs = _load(p, out)
        tok[p["arm"]] = tok.get(p["arm"], 0) + int(meta.get("total_gen_tokens") or 0)
        sec[p["arm"]] = sec.get(p["arm"], 0.0) + float((out / "wall_s.txt").read_text())
        per_doc += [r.get("gen_tokens") or 0 for r in recs]
        measured = {a: sec[a] / tok[a] for a in tok if tok[a]}
        tpd = float(np.mean(per_doc)) if per_doc else TOK_PER_DOC
        print("  measured: " + ", ".join(f"{a} {1e3 * v:.0f} ms/token" for a, v in measured.items())
              + f", {tpd:.0f} new tokens/document | remaining ~{_projection(parts[k + 1:], measured, tpd):.1f} h")
    if failed:
        print("\nflagged:\n  " + "\n  ".join(failed))
    return failed


# ----------------------------------------------------------------- judge (vLLM, harness) and score (harness)
def collect(arm, gate, smoke_part=None):
    # One generations file per arm: the verified parts' {"id", "output"} records, in id order.
    parts = [smoke_part] if smoke_part else [p for p in plan() if p["arm"] == arm]
    recs, metas, missing = [], [], []
    for p in parts:
        out = _restore(p, part_dir(p, smoke=bool(smoke_part)))
        ok, fails, _ = verify(p, out, max_new=SMOKE_NEW if smoke_part else None, quiet=True) \
            if out.is_dir() else (False, ["not generated yet"], [])
        if not ok:
            missing.append(f"{p['type']} [{p['start']}, {p['end']}) {arm}: {fails[0] if fails else '?'}")
            continue
        meta, rs = _load(p, out)
        metas.append(meta)
        recs += rs
    recs.sort(key=lambda r: r["id"])
    path = arm_dir(arm, gate, bool(smoke_part)) / (f"generations_{SETTING}_n{int(N_PER_TYPE)}.jsonl"
                                                   if not smoke_part else "generations_smoke.jsonl")
    path.parent.mkdir(parents=True, exist_ok=True)
    body = "".join(json.dumps({"id": r["id"], "output": r["output"]}, ensure_ascii=False) + "\n" for r in recs)
    if not path.is_file() or path.read_text() != body:      # rewrite only on change: keeps verdicts up to date
        path.write_text(body)
    return path, recs, metas, missing


def judge(gen_paths, tag):
    cmd = [JUDGE_PY, "-m", "longgenbench_eval.judge", "--setting", SETTING, "--data", DATA_DIR,
           "--model", JUDGE_MODEL, "--prompt", JUDGE_PROMPT, "--generations", *map(str, gen_paths)]
    if JUDGE_GREEDY:
        cmd.append("--greedy")
    env = {k: v for k, v in os.environ.items() if not k.startswith(("PYTHON", "VIRTUAL_ENV"))}
    env.update(PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    t0 = time.time()
    log = OUT_ROOT / f"judge_{tag}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    rc, tail = _stream(cmd, log, ("selftest", "SELFTEST", "judged ", "judge calls", "up to date", "Error", "Traceback"),
                       env=env)
    if rc:
        print(tail)
        raise SystemExit(f"the judge failed (exit {rc}); full log: {log}")
    print(f"  judge done in {(time.time() - t0) / 60:.1f} min (engine start included)")


def method_table(rows):
    hdr = (f"{'arm':14s} {'docs':>5s} {'gate':>12s} {'read':>6s} {'KVbytes<=':>9s} {'int2 used':>9s} "
           f"{'keys dropped':>12s} {'new tok':>8s} {'cap%':>5s} {'ms/tok*':>7s} {'peak GiB':>8s}")
    lines = [hdr, "-" * len(hdr)]
    for arm, (recs, metas) in rows.items():
        n = len(recs)
        verdicts = {(m.get("read_gate") or {}).get("verdict") for m in metas}
        rd = [m["read_gate"]["read_fraction"] for m in metas if (m.get("read_gate") or {}).get("read_fraction")]
        kvb = [r["kv_bytes_fraction_upper"] for r in recs if r.get("kv_bytes_fraction_upper") is not None]
        tg = sum(r.get("gen_tokens") or 0 for r in recs)
        ms = [r["decode_ms_per_token"] for r in recs if r.get("decode_ms_per_token") is not None]
        lines.append(f"{arm:14s} {n:>5d} {'/'.join(sorted(map(str, verdicts))):>12s} "
                     f"{(f'{np.mean(rd):.3f}' if rd else '-'):>6s} {(np.mean(kvb) if kvb else 1.0):>9.3f} "
                     f"{sum(bool(r.get('int2_used')) for r in recs) if kvb else '-':>9} "
                     f"{sum(bool(r.get('evicted')) for r in recs) if kvb else '-':>12} {tg / max(n, 1):>8.0f} "
                     f"{100 * np.mean([bool(r.get('hit_token_cap')) for r in recs]):>5.0f} "
                     f"{(f'{np.median(ms):.0f}' if ms else '-'):>7s} "
                     f"{max((float(m.get('peak_mem_gib_max') or 0) for m in metas), default=0):>8.2f}")
    lines.append("* ms/tok: median over documents of each one's decode time per token after its first token (HF generate,\n"
                 "  B = 1, evictions included): context for this run, not a perf-suite number.\n"
                 "  KVbytes<=: upper bound on the KV bytes held at the end of each document over the full cache's.")
    return "\n".join(lines)


def judge_and_score(smoke=False):
    check_knobs()
    collected, missing = {}, []
    for arm, gate in arms():
        sp = None
        if smoke:
            sp = dict(type=SMOKE_TYPE, slug=D.type_slug(SMOKE_TYPE), start=0, end=1, ids=type_ids(SMOKE_TYPE)[:1],
                      arm=arm, gate=gate)
        path, recs, metas, miss = collect(arm, gate, sp)
        missing += miss
        if recs:
            collected[arm] = (path, recs, metas)
    if not collected:
        raise SystemExit("nothing verified to judge yet: run step 11 first")
    tag = "smoke" if smoke else f"{SETTING}_n{int(N_PER_TYPE)}"
    print(f"judging {sum(len(c[1]) for c in collected.values())} documents with {JUDGE_MODEL} under vLLM "
          f"({'greedy' if JUDGE_GREEDY else 'the paper sampling'}, prompt '{JUDGE_PROMPT}') ...")
    judge([c[0] for c in collected.values()], tag)
    runs = [(arm, str(c[0]), None) for arm, c in collected.items()]
    if "fullkv" in collected:                       # deltas are reported against the first run
        runs = [r for r in runs if r[0] == "fullkv"] + [r for r in runs if r[0] != "fullkv"]
    res = SC.score(SETTING, DATA_DIR, runs)
    table = SC.format_table(res)
    method = method_table({arm: (c[1], c[2]) for arm, c in collected.items()})
    head = (f"LongGenBench {SETTING}, {len(res['ids'])} documents ({'smoke, ' + str(SMOKE_NEW) + ' new tokens' if smoke else str(int(N_PER_TYPE)) + ' per type'}), "
            f"decoding {DECODING}, judge {JUDGE_MODEL} ({JUDGE_PROMPT}{', greedy' if JUDGE_GREEDY else ''})\n"
            f"ours: budget {CACHE_BUDGET}, window {WINDOW_SIZE}, gate {GATE_RATIO}, sink {NUM_SINK}, "
            f"local {LOCAL_WINDOW}, int2 share {QUANT_RATIO} ({QUANT_MODE}), {DTYPE}; commit {_git_head()[:8]}, {_gpu_name()}")
    text = head + "\n\n" + table + "\n\n" + method
    if missing:
        text += "\n\nNOT scored (not generated yet, or did not verify; run step 11 again):\n  " + "\n  ".join(missing)
    print("\n=================== LongGenBench SCORE (the paper's metrics) ===================")
    print(text)
    out = OUT_ROOT / (f"SCORE_{tag}_{DECODING}_{op_tag()}_g{GATE_RATIO}.txt")
    out.write_text(text + "\n")
    out.with_suffix(".json").write_text(json.dumps(res, indent=2) + "\n")
    _to_drive(out)
    _to_drive(out.with_suffix(".json"))
    for g, _, _ in collected.values():
        base = str(g)[:-len(".jsonl")]
        for f in (g, Path(base + ".verdicts.jsonl"), Path(base + ".verdicts.meta.json")):
            if f.is_file():
                _to_drive(f)
    print("\nwritten to", out)
    return res


check_knobs()
print("helpers loaded | arms:", [a for a, _ in arms()], "| generation parts:", len(plan()), "| GPU:", _gpu_name(),
      "| commit:", _git_head()[:8])

## 10. Smoke run (~15 min). Must print `SMOKE OK`
Before any long GPU run, this prints our cache's geometry at your settings for the planned prompts, and refuses a
setting with no int2 tier.

Then it runs the whole pipeline once, small:
* one Menu Week document, generated by our method with up to 2,048 new tokens (that document has instructions on
  weeks 3 and 4, so the judge gets real entries to score);
* verified, with the gate verdict, and with the int2 tier used and keys dropped;
* then judged by the paper's judge under vLLM, and scored.

The score of a 2,048-token document means nothing; this checks the plumbing.

In [ ]:
check_knobs()
geometry()
ok_all = True
for arm, gate in arms():
    p = dict(type=SMOKE_TYPE, slug=D.type_slug(SMOKE_TYPE), start=0, end=1, ids=type_ids(SMOKE_TYPE)[:1], arm=arm, gate=gate)
    out = part_dir(p, smoke=True)
    print(f"\n---- smoke generation: {arm} ----")
    run_part(p, out, max_new=SMOKE_NEW)
    ok_all &= report_part(p, out, max_new=SMOKE_NEW)
    if arm != "fullkv":
        r = _load(p, out)[1][0]
        print(f"  int2 tier used: {r.get('int2_used')}   keys dropped: {r.get('evicted')}   "
              f"(fp16 holds {r.get('fp_tokens_cap')} keys, both tiers {r.get('retained_tokens_cap')})")
        ok_all &= bool(r.get("int2_used")) and bool(r.get("evicted"))
    r = _load(p, out)[1][0]
    print(f"  time: first token {r.get('first_token_s')} s (prefill plus every compile and tile search a fresh process "
          f"does; later processes reuse the on-disk Triton/Inductor caches), then {r.get('decode_ms_per_token')} ms/token "
          f"({r.get('decode_ms_per_token_last_half_median')} ms typical in the second half)")
assert ok_all, "smoke generation did not verify: do not start step 11"
print("\n---- smoke judge + score ----")
res = judge_and_score(smoke=True)
assert all(res["runs"][a]["ALL"]["judged"] > 0 for a in res["runs"]), "the judge scored no instruction"
print("\nSMOKE OK")

## 11. Generate, judge, score
For each type and chunk of documents (and arm, if you enabled more than ours), a part is generated, verified and
copied to Drive. After each part, the remaining time is re-projected from the measured speed. A part that fails a
check or runs out of GPU memory is flagged, and the next part runs.

When generation is done, the cell judges every verified document in one vLLM pass and prints the score. Run this cell
again at any point: verified parts are reused, verdicts that are already up to date are reused, and only what is
missing runs.

Expected and harmless in the output: `operation scheduled before its operands`, `tl.where with a non-boolean
condition`, the TF32 note, `failed during evaluate_expr(u0 <= 0 ...)`, and transformers' generation-config notes.

In [ ]:
FAILED = generate_all()
RESULT = judge_and_score()

## 12. Score again (e.g. after a restart, or after changing `JUDGE_*`)
After a session restart, run steps 7–9 first. On a brand-new runtime, run steps 1–5 and 7–9: the parts come back from
Drive. Only the judge runs again, and only if its settings or the generations changed.

In [ ]:
RESULT = judge_and_score()